# cet-perch — audio cache builder

Reads the curated training-set list (`training_set_composition.csv`) and the
master metadata parquet, loads raw audio for every kept window, resamples to
32 kHz, pads/crops to exactly 160 000 samples (5 s), and writes:

- `X_audio.npy`       — `(N, 160000)` float32, mmap-friendly row-major
- `meta_train.parquet` — aligned metadata with `audio_row`, `group_key`,
  `dataset`, `coarse_class` columns

These two files are the sole inputs required by `cet_perch_finetune_run.ipynb`.

---

## Label mapping

- If `label_t2 == 'background'`  → `coarse_class = 'background'`
- Otherwise                      → `coarse_class = label_t4` (species level)

## Audio loading

All datasets here use **Pattern A** — on-the-fly load from WAV:
```
librosa.load(wav_path, sr=None, offset=offset_s, duration=5.0, mono=True)
→ soxr.resample(audio, native_sr, 32000)
→ pad / crop to 160000 samples
```
Datasets that require special handling (FREMANTLE, MONISH, Watkins pickled
windows) are handled in a separate notebook.


## 0. Imports

In [2]:
import os, warnings, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import soxr
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED      = 42
TARGET_SR = 32_000
AUDIO_LEN = 160_000   # 5 s @ 32 kHz

np.random.seed(SEED)
print("Imports OK")

Imports OK


## 1. Paths — edit these to match your filesystem

In [ ]:
# ── Inputs ────────────────────────────────────────────────────────────────────

# Master metadata (all datasets, all windows)
META_ALL_PATH = Path('/data2/mromaniuc/cet-det/alltogether/full_exploration/dim_red/projector_input/meta_all.parquet')

# ── Outputs ───────────────────────────────────────────────────────────────────
OUT_DIR            = Path('/data2/mromaniuc/cet-det/cet_perchv2')
AUDIO_CACHE        = OUT_DIR / 'X_audio.npy'
META_OUT           = OUT_DIR / 'meta_train.parquet'
# Full-column filtered selection saved after Step 2 (all meta_all columns kept)
META_SELECTION_OUT = OUT_DIR / 'meta_train_selection.parquet'

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset root folders ──────────────────────────────────────────────────────
# Keys must match the 'dataset' column values in meta_all exactly.
# wav_path in meta_all is already absolute — this dict is a fallback only.
DATASET_ROOTS = {
    'Adriatic_Sea'      : Path('/data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA/Audio_data'),
    'ALNITAK_CAVANILLES': Path('/data2/mromaniuc/cet-det/datasets/ALNITAK_CAVANILLES'),
    'DCLDE_2026'        : Path('/data2/mromaniuc/cet-det/datasets/DCLDE_2026/audio'),
    'DOLPHINFREE'       : Path('/data2/mromaniuc/cet-det/datasets/DOLPHINFREE/Single_hydrophone/inputs/Audio_data'),
    'DRYAD'             : Path('/data2/mromaniuc/cet-det/datasets/DRYAD/Audio'),
    'ECOSS_testtrain'   : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/testingtraining_sounds'),
    'ECOSS_annot'       : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/annotated_sounds'),
    'ECOSS_enhanced'    : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/enhanced4AI_sounds'),
    'OLTREMARE'         : Path('/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS'),
    # Pattern-B datasets — handled separately
    'FREMANTLE'         : None,
    'MONISH'            : None,
    'WATKINS'           : None,
}

# ── Sanity check ──────────────────────────────────────────────────────────────
print(f"META_ALL_PATH : {'OK' if META_ALL_PATH.exists() else 'MISSING'}  {META_ALL_PATH}")
print(f"OUT_DIR       : {OUT_DIR}")
print()
for ds, root in DATASET_ROOTS.items():
    if root is None:
        print(f"  {ds:20s}  Pattern-B (skip here)")
    else:
        status = 'OK' if root.exists() else 'MISSING'
        print(f"  {ds:20s}  {status}  {root}")

## 2. Per-dataset selection

Each dataset is filtered and sampled independently.
Output is `meta_kept`, which feeds Steps 3–10 unchanged.

| Dataset | Background kept | Mammal / signal kept |
|---|---|---|
| Adriatic_Sea | all bg (238) | TT whistles 147, TT clicks 73 |
| ALNITAK_CAVANILLES | 1 854 non-mammal | all mammals |
| DCLDE_2026 | scripps 300, orcasound 100, onc 400, dfo_wdlp+vfpa 100 | Orcinus orca 887 |
| DOLPHINFREE | — | Delphinus delphis all (1 150) |
| DRYAD | — | TT whistles 73 × 3 sources |
| ECOSS_annot | anthropogenic all (286) | delphinids all |
| ECOSS_testtrain | hammer 8, rainfall 32, passenger 400, tanker 400, tug 100, cargo 400, glider 400 | all mammals |
| ECOSS_enhanced | hammer 25 | all biological |
| OLTREMARE | — | whistle 73, click 73, burst_pulse 73, feeding_buzz 73 |

**Balaenoptera acutorostrata is excluded from every dataset.**

In [7]:
print("Loading master metadata...")
meta_all = pd.read_parquet(META_ALL_PATH)
print(f"  meta_all: {len(meta_all):,} rows\n")

def sample_n(df, n, seed=SEED):
    """Return n random rows; if fewer are available, return all."""
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).copy()

kept_frames = []

# ── Adriatic_Sea ──────────────────────────────────────────────────────────────
a = meta_all[meta_all['dataset'] == 'Adriatic_Sea']
kept_frames.append(a[a['label_t2'] == 'background'])                               # all bg (238)
kept_frames.append(sample_n(                                                        # TT whistles (147)
    a[(a['label_t4'] == 'Tursiops_truncatus') &
      (a['label_t3'].isin(['whistle', 'click+whistle']))], 147))
kept_frames.append(sample_n(                                                        # TT clicks (73)
    a[(a['label_t4'] == 'Tursiops_truncatus') &
      (a['label_t3'] == 'click')], 73))

# ── ALNITAK_CAVANILLES ────────────────────────────────────────────────────────
al = meta_all[meta_all['dataset'] == 'ALNITAK_CAVANILLES']
kept_frames.append(sample_n(al[al['label_t1'] == 'non_mammal'], 1854))             # 1854 bg
kept_frames.append(al[al['label_t1'] == 'mammal'])                                 # all mammals (854)

# ── DCLDE_2026 ────────────────────────────────────────────────────────────────
# Deployment folder is the path component right after /audio/.
# scripps=SIO (towed array), orcasound=ORCASOUND (boat shallow 100m),
# onc=ONC (motorized boat deep), dfo_wdlp+vfpa=DFO_WDLP/JASCVO_VFPA (ship cargo).
dc = meta_all[meta_all['dataset'] == 'DCLDE_2026'].copy()
dc['_dep'] = dc['wav_path'].astype(str).str.extract(r'/audio/([^/]+)/')[0]
dc_bg = dc[dc['label_t2'] == 'background']
for _dep, _n in [('scripps', 300), ('orcasound', 100), ('onc', 400)]:
    kept_frames.append(sample_n(dc_bg[dc_bg['_dep'] == _dep].drop(columns='_dep'), _n))
kept_frames.append(sample_n(                                                        # ship cargo (100)
    dc_bg[dc_bg['_dep'].isin(['dfo_wdlp', 'vfpa'])].drop(columns='_dep'), 100))
kept_frames.append(sample_n(                                                        # Orcinus orca (887)
    dc[dc['label_t4'] == 'Orcinus_orca'].drop(columns='_dep'), 887))

# ── DOLPHINFREE ───────────────────────────────────────────────────────────────
dol = meta_all[meta_all['dataset'] == 'DOLPHINFREE']
kept_frames.append(dol[dol['label_t4'] == 'Delphinus_delphis'])                    # all 1150 whistles

# ── DRYAD ─────────────────────────────────────────────────────────────────────
dry = meta_all[meta_all['dataset'] == 'DRYAD']
for src in ['Oceanographic', 'IMMS', 'SWFSC']:
    kept_frames.append(sample_n(
        dry[(dry['subdataset'] == src) &
            (dry['label_t4'] == 'Tursiops_truncatus') &
            (dry['label_t3'] == 'whistle')], 73))                                   # 73 TT whistles each

# ── ECOSS_annot ───────────────────────────────────────────────────────────────
ea = meta_all[meta_all['dataset'] == 'ECOSS_annot']
kept_frames.append(ea[ea['label_t2'] == 'anthropogenic'])                           # 286 bg (all)
kept_frames.append(ea[ea['label_t1'] == 'mammal'])                                  # delphinids (all)

# ── ECOSS_testtrain ───────────────────────────────────────────────────────────
et = meta_all[meta_all['dataset'] == 'ECOSS_testtrain']
for _lbl, _n in [('Anthropogenic_Hammer',  8),
                 ('Geophonic_Rainfall',   32),
                 ('Ship_Passengership',  400),
                 ('Ship_Tanker',         400),
                 ('Ship_Tug',            100),
                 ('Ship_Cargo',          400),
                 ('Glider_Ambient',      400)]:
    kept_frames.append(sample_n(et[et['label'] == _lbl], _n))
kept_frames.append(et[et['label_t1'] == 'mammal'])                                  # all mammals (127)

# ── ECOSS_enhanced ────────────────────────────────────────────────────────────
ee = meta_all[meta_all['dataset'] == 'ECOSS_enhanced']
kept_frames.append(sample_n(ee[ee['label'] == 'Hammer'], 25))                       # hammer bg (all 25)
kept_frames.append(ee[ee['label_t1'] == 'mammal'])                                  # all biological (143)

# ── OLTREMARE ─────────────────────────────────────────────────────────────────
olt = meta_all[meta_all['dataset'] == 'OLTREMARE']
for sig in ['whistle', 'click', 'burst_pulse', 'feeding_buzz']:
    kept_frames.append(sample_n(olt[olt['label_t3'] == sig], 73))                   # 73 each

# ── Assemble, clean, report ───────────────────────────────────────────────────
meta_kept = pd.concat(kept_frames, ignore_index=True)

# Exclude Balaenoptera acutorostrata from all datasets
_bala = meta_kept['label_t4'].astype(str).str.lower().str.contains(
    r'balaenoptera.acutorostrata', na=False)
if _bala.any():
    print(f"Removing {_bala.sum()} Balaenoptera acutorostrata rows")
    meta_kept = meta_kept[~_bala]

# global_idx is dataset-local (resets per dataset), so dedup on the pair.
# This also catches any accidental repeated pool inclusions.
_n_before = len(meta_kept)
meta_kept = meta_kept.drop_duplicates(
    subset=['dataset', 'global_idx']).reset_index(drop=True)
if _n_before != len(meta_kept):
    print(f"Removed {_n_before - len(meta_kept)} duplicate windows")

print(f"Total kept windows: {len(meta_kept):,}\n")
print("Per-dataset summary:")
for _ds, _grp in meta_kept.groupby('dataset'):
    print(f"  {_ds:40s}  {len(_grp):>6,}")

Loading master metadata...
  meta_all: 247,630 rows

Total kept windows: 9,401

Per-dataset summary:
  ALNITAK_CAVANILLES                         2,708
  Adriatic_Sea                                 458
  DCLDE_2026                                 1,787
  DOLPHINFREE                                1,150
  DRYAD                                        219
  ECOSS_annot                                  752
  ECOSS_enhanced                               168
  ECOSS_testtrain                            1,867
  OLTREMARE                                    292


In [14]:
META_SELECTION_OUT = Path('/data2/mromaniuc/cet-det/cet_perchv2/meta_selection.parquet')

SIGNAL_TYPES = {'click', 'whistle', 'click+whistle', 'burst_pulse',
                'feeding_buzz', 'unknown', 'echolocation'}

def get_coarse(row):
    t2 = str(row.get('label_t2', '') or '').strip().lower()
    t3 = str(row.get('label_t3', '') or '').strip().lower()
    t4 = row.get('label_t4', None)

    if t2 in ('background', 'anthropogenic', 'geophonic'):
        return 'background'
    if pd.notna(t4) and str(t4).strip() not in ('', 'None'):
        return str(t4).strip()
    if t2 in ('odontocete', 'delphinid') or t3 in SIGNAL_TYPES or t3 == '':
        return 'Delphinids'
    return t3 if t3 else 'unknown'

meta_kept['coarse_class'] = meta_kept.apply(get_coarse, axis=1)

# Sanity check before saving
print("Coarse class distribution:")
for cls, n in meta_kept['coarse_class'].value_counts().items():
    print(f"  {cls:40s}  {n:>6,}")
print(f"\nunknown rows : {(meta_kept['coarse_class'] == 'unknown').sum()}")
assert (meta_kept['coarse_class'] == 'unknown').sum() == 0, \
    "Fix unknown rows before saving"

meta_kept.to_parquet(META_SELECTION_OUT, index=False)
print(f"\nWritten: {META_SELECTION_OUT}")
print(f"  Rows    : {len(meta_kept):,}")
print(f"  Columns : {len(meta_kept.columns)}")

Coarse class distribution:
  background                                 5,043
  Delphinus_delphis                          1,190
  Orcinus_orca                                 924
  Tursiops_truncatus                           835
  Delphinidae_unknown                          508
  Delphinids                                   416
  Physeter_macrocephalus                       214
  Globicephala_melas                           167
  Stenella_coeruleoalba                         59
  Grampus_griseus                               45

unknown rows : 0

Written: /data2/mromaniuc/cet-det/cet_perchv2/meta_selection.parquet
  Rows    : 9,401
  Columns : 64


In [15]:
print("\nGlobicephala_melas breakdown by dataset:")
gm = meta_kept[meta_kept['label_t4'] == 'Globicephala_melas']
print(f"  Total: {len(gm)}")
print(gm['dataset'].value_counts().to_string())


Globicephala_melas breakdown by dataset:
  Total: 167
dataset
ALNITAK_CAVANILLES    141
ECOSS_testtrain        26


In [17]:
print("\nGlobicephala_melas in meta_all (all datasets):")
gm_all = meta_all[meta_all['label_t4'] == 'Globicephala_melas']
print(f"  Total: {len(gm_all)}")
print(gm_all['dataset'].value_counts().to_string())

print("\nALNITAK_CAVANILLES label_t4 distribution:")
print(meta_all[meta_all['dataset'] == 'ALNITAK_CAVANILLES']['label_t4'].value_counts().to_string())

print("\nALNITAK_CAVANILLES label_t1 distribution:")
print(meta_all[meta_all['dataset'] == 'ALNITAK_CAVANILLES']['label_t1'].value_counts().to_string())


Globicephala_melas in meta_all (all datasets):
  Total: 721
dataset
MONISH                410
WATKINS               144
ALNITAK_CAVANILLES    141
ECOSS_testtrain        26

ALNITAK_CAVANILLES label_t4 distribution:
label_t4
Physeter_macrocephalus    179
Globicephala_melas        141
Tursiops_truncatus         58
Grampus_griseus            45
Stenella_coeruleoalba      15

ALNITAK_CAVANILLES label_t1 distribution:
label_t1
non_mammal    4348
mammal         854


In [20]:
import pickle

WATKINS_PKL = Path('/data2/mromaniuc/cet-det/models/perch_v2/WATKINS/windows_df.pkl')
MONISH_PKL  = Path('/data2/mromaniuc/cet-det/models/perch_v2/MONISH/models/perch_v2/LAST/windows_df.pkl')

with open(WATKINS_PKL, 'rb') as f:
    watkins_df = pickle.load(f)
with open(MONISH_PKL, 'rb') as f:
    monish_df  = pickle.load(f)

# Add dataset column and coarse_class, drop Balaenoptera acutorostrata
watkins_df['dataset'] = 'Watkins'
monish_df['dataset']  = 'MONISH'

pattern_b = pd.concat([watkins_df, monish_df], ignore_index=True)

# Drop B. acutorostrata
_bala = pattern_b['label'].str.lower().str.contains('balaenoptera_acutorostrata', na=False)
print(f"Dropping {_bala.sum()} Balaenoptera acutorostrata rows")
pattern_b = pattern_b[~_bala].reset_index(drop=True)

# coarse_class is just label for these (already species-level)
pattern_b['coarse_class'] = pattern_b['label']

# group_key — source_file identifies the recording
pattern_b['group_key'] = pattern_b['dataset'] + '_' + pattern_b['source_file']

print(f"\nPattern-B windows: {len(pattern_b):,}")
print("\nPer dataset + class:")
for (ds, cls), grp in pattern_b.groupby(['dataset', 'coarse_class']):
    print(f"  {ds:10s}  {cls:35s}  {len(grp):>4,}")

Dropping 17 Balaenoptera acutorostrata rows

Pattern-B windows: 2,477

Per dataset + class:
  MONISH      Balaenoptera_physalus                  29
  MONISH      Globicephala_melas                    410
  MONISH      Grampus_griseus                       113
  MONISH      Orcinus_orca                           37
  MONISH      Tursiops_truncatus                     28
  Watkins     Balaenoptera_physalus                 510
  Watkins     Delphinus_delphis                     102
  Watkins     Globicephala_melas                    144
  Watkins     Grampus_griseus                        81
  Watkins     Orcinus_orca                           39
  Watkins     Physeter_macrocephalus                875
  Watkins     Stenella_coeruleoalba                  85
  Watkins     Tursiops_truncatus                     24


In [21]:
# ── 1. Load existing meta_selection and append Pattern-B metadata ─────────────
meta_selection = pd.read_parquet(META_SELECTION_OUT)
print(f"meta_selection before: {len(meta_selection):,} rows")

# Build minimal metadata rows for pattern-B (no wav_path — audio is in pickle)
pb_meta = pattern_b[['dataset', 'coarse_class', 'group_key', 'label',
                      'source_file', 'window_idx', 'window_kind']].copy()

# Add columns present in meta_selection but absent in pattern-B (fill with None)
for col in meta_selection.columns:
    if col not in pb_meta.columns:
        pb_meta[col] = None

# Reorder to match
pb_meta = pb_meta[meta_selection.columns]

meta_combined = pd.concat([meta_selection, pb_meta], ignore_index=True)
print(f"meta_selection after : {len(meta_combined):,} rows")

print("\nFull coarse_class distribution:")
for cls, n in meta_combined['coarse_class'].value_counts().items():
    print(f"  {cls:40s}  {n:>6,}")

print("\nFull dataset distribution:")
for ds, n in meta_combined['dataset'].value_counts().items():
    print(f"  {ds:25s}  {n:>6,}")

meta_selection before: 9,401 rows
meta_selection after : 11,878 rows

Full coarse_class distribution:
  background                                 5,043
  Delphinus_delphis                          1,292
  Physeter_macrocephalus                     1,089
  Orcinus_orca                               1,000
  Tursiops_truncatus                           887
  Globicephala_melas                           721
  Balaenoptera_physalus                        539
  Delphinidae_unknown                          508
  Delphinids                                   416
  Grampus_griseus                              239
  Stenella_coeruleoalba                        144

Full dataset distribution:
  ALNITAK_CAVANILLES          2,708
  ECOSS_testtrain             1,867
  Watkins                     1,860
  DCLDE_2026                  1,787
  DOLPHINFREE                 1,150
  ECOSS_annot                   752
  MONISH                        617
  Adriatic_Sea                  458
  OLTREMARE          

In [22]:
# Merge Delphinidae_unknown into Delphinids
meta_combined['coarse_class'] = meta_combined['coarse_class'].replace(
    'Delphinidae_unknown', 'Delphinids'
)

# Fix in meta_selection parquet on disk too
meta_selection_fixed = pd.read_parquet(META_SELECTION_OUT)
meta_selection_fixed['coarse_class'] = meta_selection_fixed['coarse_class'].replace(
    'Delphinidae_unknown', 'Delphinids'
)
meta_selection_fixed.to_parquet(META_SELECTION_OUT, index=False)
print(f"meta_selection.parquet updated")

print("\nFull coarse_class distribution after merge:")
for cls, n in meta_combined['coarse_class'].value_counts().items():
    print(f"  {cls:40s}  {n:>6,}")

meta_selection.parquet updated

Full coarse_class distribution after merge:
  background                                 5,043
  Delphinus_delphis                          1,292
  Physeter_macrocephalus                     1,089
  Orcinus_orca                               1,000
  Delphinids                                   924
  Tursiops_truncatus                           887
  Globicephala_melas                           721
  Balaenoptera_physalus                        539
  Grampus_griseus                              239
  Stenella_coeruleoalba                        144


In [24]:
# Drop mixed-species labels from Delphinids
mixed_labels = {
    'gloMel_clicks+gloMel_whistles+turTru_clicks+turTru_whistles',
    'gloMel_clicks+turTru_clicks'
}

drop_mask = (meta_combined['coarse_class'] == 'Delphinids') & \
            (meta_combined['label'].isin(mixed_labels))

print(f"Dropping {drop_mask.sum()} mixed-species rows from Delphinids")
meta_combined = meta_combined[~drop_mask].reset_index(drop=True)

# Fix in meta_selection.parquet on disk too
meta_sel = pd.read_parquet(META_SELECTION_OUT)
drop_mask_sel = (meta_sel['coarse_class'] == 'Delphinids') & \
                (meta_sel['label'].isin(mixed_labels))
print(f"Dropping {drop_mask_sel.sum()} rows from meta_selection.parquet")
meta_sel = meta_sel[~drop_mask_sel].reset_index(drop=True)
meta_sel.to_parquet(META_SELECTION_OUT, index=False)

print(f"\nmeta_combined: {len(meta_combined):,} rows")
print("\nDelphinids after cleanup:")
delph = meta_combined[meta_combined['coarse_class'] == 'Delphinids']
print(f"  Total: {len(delph):,}")
print(delph.groupby(['dataset', 'label']).size().to_string())

Dropping 109 mixed-species rows from Delphinids
Dropping 109 rows from meta_selection.parquet

meta_combined: 11,769 rows

Delphinids after cleanup:
  Total: 815
dataset             label                 
ALNITAK_CAVANILLES  delphi_clicks             304
                    delphi_whistles             3
ECOSS_annot         Odontocetes_Delphinids    466
ECOSS_enhanced      Odontocetes_Delphinids     42


In [25]:
# ── 2. Build X_audio.npy ──────────────────────────────────────────────────────
import librosa, soxr
from tqdm.auto import tqdm

AUDIO_CACHE = OUT_DIR / 'X_audio.npy'
N           = len(meta_combined)

print(f"Allocating X_audio: ({N}, {AUDIO_LEN}) float32  "
      f"= {N * AUDIO_LEN * 4 / 1e9:.2f} GB")

X_audio = np.lib.format.open_memmap(
    str(AUDIO_CACHE), mode='w+', dtype=np.float32, shape=(N, AUDIO_LEN)
)

# Build a lookup for pattern-B audio: (dataset, source_file, window_idx) → audio
pb_lookup = {
    (row['dataset'], row['source_file'], row['window_idx']): row['audio']
    for _, row in pattern_b.iterrows()
}

errors = []
for i, row in tqdm(meta_combined.iterrows(), total=N, desc='Building cache'):
    try:
        ds = row['dataset']
        if ds in ('Watkins', 'MONISH'):
            # Pattern-B: pull from pickle lookup
            key = (ds, row['source_file'], row['window_idx'])
            X_audio[i] = pb_lookup[key]
        else:
            # Pattern-A: load from WAV
            wav  = row['wav_path']
            off  = float(row['offset_s']) if pd.notna(row['offset_s']) else 0.0
            audio_native, native_sr = librosa.load(
                wav, sr=None, offset=off, duration=5.0, mono=True)
            if native_sr != TARGET_SR:
                audio = soxr.resample(audio_native, native_sr, TARGET_SR)
            else:
                audio = audio_native
            if len(audio) < AUDIO_LEN:
                audio = np.pad(audio, (0, AUDIO_LEN - len(audio)))
            X_audio[i] = audio[:AUDIO_LEN].astype(np.float32)
    except Exception as e:
        errors.append((i, row.get('dataset'), row.get('wav_path'), str(e)))
        X_audio[i] = np.zeros(AUDIO_LEN, dtype=np.float32)

X_audio.flush()
print(f"\nDone.  {N - len(errors):,} OK  |  {len(errors)} errors")
if errors:
    print("\nFirst 10 errors:")
    for i, ds, wav, msg in errors[:10]:
        print(f"  row={i}  {ds}  {wav}  →  {msg}")

# ── 3. Add audio_row and save final meta_train.parquet ────────────────────────
meta_combined = meta_combined.reset_index(drop=True)
meta_combined['audio_row'] = meta_combined.index.astype(np.int64)

META_TRAIN_OUT = OUT_DIR / 'meta_train.parquet'
meta_combined.to_parquet(META_TRAIN_OUT, index=False)

# Final assertions
assert len(meta_combined) == X_audio.shape[0]
assert (meta_combined['audio_row'].values == np.arange(len(meta_combined))).all()

print(f"\nWritten: {META_TRAIN_OUT}")
print(f"  Rows          : {len(meta_combined):,}")
print(f"  X_audio shape : {X_audio.shape}")
print(f"\nReady for cet_perch_finetune_run.ipynb ✓")

Allocating X_audio: (11769, 160000) float32  = 7.53 GB


Building cache:   0%|          | 0/11769 [00:00<?, ?it/s]


Done.  11,769 OK  |  0 errors

Written: /data2/mromaniuc/cet-det/cet_perchv2/meta_train.parquet
  Rows          : 11,769
  X_audio shape : (11769, 160000)

Ready for cet_perch_finetune_run.ipynb ✓


## Validate the cache

In [26]:
print("Validating cache...")
X_audio_r = np.load(str(AUDIO_CACHE), mmap_mode='r')
print(f"  Shape  : {X_audio_r.shape}")
print(f"  Dtype  : {X_audio_r.dtype}")
print(f"  Global min/max: {X_audio_r.min():.4f} / {X_audio_r.max():.4f}")

print("\nPer-class amplitude stats (RMS):")
for cls in sorted(meta_combined['coarse_class'].unique()):
    idx = meta_combined[meta_combined['coarse_class'] == cls].index.tolist()
    sample_idx = idx[:min(50, len(idx))]
    rms_vals = np.sqrt(np.mean(X_audio_r[sample_idx] ** 2, axis=1))
    print(f"  {cls:40s}  n={len(idx):>5,}  "
          f"rms_mean={rms_vals.mean():.5f}  rms_std={rms_vals.std():.5f}")

zero_rows = np.where(~X_audio_r.any(axis=1))[0]
print(f"\nAll-zero rows (failed loads): {len(zero_rows)}")
if len(zero_rows) > 0:
    print(f"  Row indices: {zero_rows[:20]}")

Validating cache...
  Shape  : (11769, 160000)
  Dtype  : float32
  Global min/max: -1.5937 / 1.4531

Per-class amplitude stats (RMS):
  Balaenoptera_physalus                     n=  539  rms_mean=0.21268  rms_std=0.03668
  Delphinids                                n=  815  rms_mean=0.00063  rms_std=0.00016
  Delphinus_delphis                         n=1,292  rms_mean=0.01109  rms_std=0.01675
  Globicephala_melas                        n=  721  rms_mean=0.00418  rms_std=0.00894
  Grampus_griseus                           n=  239  rms_mean=0.02164  rms_std=0.04410
  Orcinus_orca                              n=1,000  rms_mean=0.00385  rms_std=0.00745
  Physeter_macrocephalus                    n=1,089  rms_mean=0.00352  rms_std=0.00324
  Stenella_coeruleoalba                     n=  144  rms_mean=0.02055  rms_std=0.02225
  Tursiops_truncatus                        n=  887  rms_mean=0.08183  rms_std=0.05614
  background                                n=5,043  rms_mean=0.03826  rms_std=0.0

## Write `meta_train.parquet`

In [28]:
# group_key is missing from meta_combined — derive it
def derive_group_key(row):
    # Pattern-B (Watkins/MONISH) — already have source_file
    if row.get('dataset') in ('Watkins', 'MONISH'):
        return f"{row['dataset']}_{row['source_file']}"
    # Pattern-A — use wav_path as unique file identifier
    wp = row.get('wav_path', None)
    if pd.notna(wp) and str(wp).strip() not in ('', 'None'):
        return str(wp).strip()
    # Fallback
    return f"{row.get('dataset','')}_{row.get('global_idx','')}"

meta_combined['group_key'] = meta_combined.apply(derive_group_key, axis=1)

print(f"Unique group_keys: {meta_combined['group_key'].nunique():,}  "
      f"(out of {len(meta_combined):,} windows)")
print("\nGroups per dataset:")
for ds, grp in meta_combined.groupby('dataset'):
    print(f"  {ds:25s}  {grp['group_key'].nunique():>5,} groups  {len(grp):>6,} windows")

Unique group_keys: 2,814  (out of 11,769 windows)

Groups per dataset:
  ALNITAK_CAVANILLES           443 groups   2,599 windows
  Adriatic_Sea                   1 groups     458 windows
  DCLDE_2026                   666 groups   1,787 windows
  DOLPHINFREE                  150 groups   1,150 windows
  DRYAD                         37 groups     219 windows
  ECOSS_annot                   94 groups     752 windows
  ECOSS_enhanced               149 groups     168 windows
  ECOSS_testtrain              537 groups   1,867 windows
  MONISH                       185 groups     617 windows
  OLTREMARE                    118 groups     292 windows
  Watkins                      434 groups   1,860 windows


In [29]:
# audio_row already set, just write meta_train.parquet
required_cols = ['audio_row', 'group_key', 'dataset', 'coarse_class']
extra_cols = [c for c in [
    'global_idx', 'wav_path', 'offset_s',
    'label', 'label_t2', 'label_t3', 'label_t4',
    'region', 'environment', 'source_file', 'window_idx',
] if c in meta_combined.columns]

meta_out = meta_combined[required_cols + extra_cols].copy()

assert (meta_out['audio_row'].values == np.arange(len(meta_out))).all(), \
    "audio_row not aligned"
assert meta_out['group_key'].notna().all(), "group_key has nulls"
assert meta_out['coarse_class'].notna().all(), "coarse_class has nulls"
assert len(meta_out) == X_audio_r.shape[0], \
    f"length mismatch: meta {len(meta_out)} vs X_audio {X_audio_r.shape[0]}"

META_TRAIN_OUT = OUT_DIR / 'meta_train.parquet'
meta_out.to_parquet(META_TRAIN_OUT, index=False)
print(f"Written: {META_TRAIN_OUT}")
print(f"  Rows    : {len(meta_out):,}")
print(f"  Columns : {list(meta_out.columns)}")
print("\nFinal class distribution:")
for cls, n in meta_out['coarse_class'].value_counts().items():
    print(f"  {cls:40s}  {n:>6,}")
print("\nFinal dataset distribution:")
for ds, n in meta_out['dataset'].value_counts().items():
    print(f"  {ds:25s}  {n:>6,}")
print("\nReady for cet_perch_finetune_run.ipynb ✓")

Written: /data2/mromaniuc/cet-det/cet_perchv2/meta_train.parquet
  Rows    : 11,769
  Columns : ['audio_row', 'group_key', 'dataset', 'coarse_class', 'global_idx', 'wav_path', 'offset_s', 'label', 'label_t2', 'label_t3', 'label_t4', 'region', 'environment', 'source_file', 'window_idx']

Final class distribution:
  background                                 5,043
  Delphinus_delphis                          1,292
  Physeter_macrocephalus                     1,089
  Orcinus_orca                               1,000
  Tursiops_truncatus                           887
  Delphinids                                   815
  Globicephala_melas                           721
  Balaenoptera_physalus                        539
  Grampus_griseus                              239
  Stenella_coeruleoalba                        144

Final dataset distribution:
  ALNITAK_CAVANILLES          2,599
  ECOSS_testtrain             1,867
  Watkins                     1,860
  DCLDE_2026                  1,787
  D

## 10. Final compatibility check with fine-tuning notebook

In [30]:
print("Final compatibility check...")

X_check    = np.load(str(AUDIO_CACHE), mmap_mode='r')
meta_check = pd.read_parquet(META_TRAIN_OUT)  # ← was META_OUT

assert X_check.shape[1] == AUDIO_LEN, \
    f"X_audio width {X_check.shape[1]} != {AUDIO_LEN}"
assert X_check.dtype == np.float32, \
    f"X_audio dtype {X_check.dtype} != float32"
assert len(meta_check) == X_check.shape[0], \
    f"Length mismatch: meta {len(meta_check)} vs X_audio {X_check.shape[0]}"
assert (meta_check['audio_row'].values == np.arange(len(meta_check))).all(), \
    "audio_row not aligned with row position"
assert set(required_cols).issubset(meta_check.columns), \
    f"Missing required columns: {set(required_cols) - set(meta_check.columns)}"

print("  X_audio.npy          OK")
print("  meta_train.parquet   OK")
print(f"  N windows            {X_check.shape[0]:,}")
print(f"  N classes            {meta_check['coarse_class'].nunique()}")
print(f"  N datasets           {meta_check['dataset'].nunique()}")
print(f"  N groups             {meta_check['group_key'].nunique():,}")
print()
print("Ready for cet_perch_finetune_run.ipynb  ✓")

Final compatibility check...
  X_audio.npy          OK
  meta_train.parquet   OK
  N windows            11,769
  N classes            10
  N datasets           11
  N groups             2,814

Ready for cet_perch_finetune_run.ipynb  ✓
